In [20]:
import sys,os

fpath = os.path.dirname(os.path.abspath("C:/Users/au646858/OneDrive - Aarhus Universitet/Documents/_repositories/sea-anemone/app/session.py"))
if fpath not in sys.path:
    sys.path.append(fpath)

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go


In [21]:
from map import Map

In [33]:
df = pd.read_csv(r"C:\Users\au646858\OneDrive - Aarhus universitet\Documents\anemone_projects\es5\site.csv")

lon_span = np.linspace(df["longitude"].min(), df["longitude"].max(), 100)
lat_span = np.linspace(df["latitude"].min(), df["latitude"].max(), 100)

x = lon_span-lon_span.mean()
y = lat_span-lat_span.mean()

r2 = x[:, None] + y[None, :]

In [34]:
m1 = Map(data=df)

In [46]:
import plotly.graph_objects as go

m1 = Map(data=df)
m1.settings.markersize=0.
fig = m1.render()

# Option 1: Density heatmap from point data
fig.add_trace(go.Densitymap(
    lat=df["latitude"],
    lon=df["longitude"],
    z=df["doi"],   # values to color by
    radius=10,              # pixel radius of influence
    colorscale="Viridis",
    opacity=1.,
    showscale=True,
))
fig

In [40]:
import plotly.graph_objects as go
import json

# Build GeoJSON grid of rectangles
features = []
z_values = []
ids = []

dlon = (lon_span[1] - lon_span[0]) / 2
dlat = (lat_span[1] - lat_span[0]) / 2

for i, lat in enumerate(lat_span):
    for j, lon in enumerate(lon_span):
        cell_id = f"{i}-{j}"
        ids.append(cell_id)
        z_values.append(float(r2[i, j]))
        features.append({
            "type": "Feature",
            "id": cell_id,
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [lon - dlon, lat - dlat],
                    [lon + dlon, lat - dlat],
                    [lon + dlon, lat + dlat],
                    [lon - dlon, lat + dlat],
                    [lon - dlon, lat - dlat],
                ]]
            }
        })

geojson = {"type": "FeatureCollection", "features": features}

In [50]:
fig = m1.render()
m1.settings.markersize=3.
fig.add_trace(go.Choroplethmap(
    geojson=geojson,
    locations=ids,
    z=z_values,
    colorscale="Turbo",
    marker=dict(opacity=1, line=dict(width=0)),
    showscale=True,
    hoverinfo="skip",       # no hover tooltip
    selectedpoints=[],       # nothing pre-selected
    unselected=dict(marker=dict(opacity=1)),  # keep appearance on selection
))
# fig.update_layout(
#     clickmode="event",       # don't select on click
# )
data_list = list(fig.data)
fig.data = [data_list[-1]] + data_list[:-1]
fig